# Restricted-input method reference
Source: gps_analysis/residence_identification/01-residence_identification_batch.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from datetime import datetime, timedelta, time
from typing import Tuple, Dict, Optional

In [ ]:
def majority_day(start: pd.Timestamp, end: pd.Timestamp):
    # Iterate day-by-day, measuring overlap with [start, end)
    cur = start.normalize()
    stop = end.normalize() + pd.Timedelta(days=1)

    best_day = start.date()
    best_dur = pd.Timedelta(0)

    while cur < stop:
        nxt = cur + pd.Timedelta(days=1)
        a = max(start, cur)
        b = min(end,   nxt)
        dur = max(pd.Timedelta(0), b - a)
        if dur > best_dur:
            best_dur = dur
            best_day = cur.date()
        cur = nxt
    return best_day

def _hour_in_range(hour: int, start: int, end: int) -> bool:
    if start < end:
        return start <= hour < end
    # wrap-around, e.g. 22..6
    return hour >= start or hour < end


def _unique_days(series: pd.Series) -> set:
    return set(series.dt.weekday.unique())

In [ ]:
import pandas as pd
from typing import Dict, Optional, Iterable


def _normalise_day_label(s: str) -> str:
    """Normalise day label (e.g. 'friday' -> 'Friday')."""
    if pd.isna(s):
        return ""
    return str(s).strip().title()


def predict_residence_and_workplace_from_tod(
    df: pd.DataFrame,
    cluster_col: str = "cluster_id",
    day_col: str = "activity_dayofweek",
    tod_col: str = "activity_tod",
    night_tods: Optional[Iterable[str]] = None,
) -> Dict[str, object]:
    """
    Predict residence and workplace clusters using categorical day-of-week and time-of-day columns.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with at least columns for cluster id, day-of-week and time-of-day.
        Example columns in your case: 'cluster_id', 'activity_dayofweek', 'activity_tod'.
    cluster_col : str
        Column name for cluster IDs.
    day_col : str
        Column name for day-of-week labels (e.g. 'Monday', 'Tuesday', ...).
    tod_col : str
        Column name for time-of-day labels (e.g. 'Morning', 'Afternoon', 'Evening', 'Night').
    night_tods : iterable[str], optional
        Which `activity_tod` labels count as "night". Default: {'Night', 'Evening'}.
        You may adjust if your labelling differs.

    Returns
    -------
    dict
        {
          "residence_scores": {cluster: score, ...},
          "work_scores": {cluster: score, ...},
          "predicted_residence": cluster_id or None,
          "predicted_workplace": cluster_id or None,
          "notes": textual notes (if any)
        }

    Behaviour
    ---------
    - Uses the same formulas:
        S_res(Cj) = 0.5*(|Days(V_j_night)|/7) + 0.5*(|V_j_night|/|V_night|)
        S_work(Cj) = 0.5*(|Days(V_j_work)|/5) + 0.5*(|V_j_work|/|V_work|)
    - Day labels are normalised (case-insensitive). Only valid weekday names are counted.
    - If |V_night| or |V_work| is zero, the frequency term is 0 and a note is added.
    - Workplace is chosen from clusters != residence.
    """
    if night_tods is None:
        night_tods = {"Night", "Evening"}

    # Validate columns
    if cluster_col not in df.columns or day_col not in df.columns or tod_col not in df.columns:
        raise ValueError(f"DataFrame must contain columns '{cluster_col}', '{day_col}', '{tod_col}'.")

    # Normalise labels
    df_proc = df[[cluster_col, day_col, tod_col]].copy()
    df_proc[day_col] = df_proc[day_col].apply(_normalise_day_label)
    df_proc[tod_col] = df_proc[tod_col].apply(lambda s: "" if pd.isna(s) else str(s).strip().title())

    # Define valid day names and weekday set
    VALID_DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    WEEKDAYS = set(VALID_DAYS[:5])  # Monday..Friday

    # Create boolean masks
    night_tods_norm = {str(x).strip().title() for x in night_tods}

    df_proc["is_night"] = df_proc[tod_col].isin(night_tods_norm)

    # Counts
    night_df = df_proc[df_proc["is_night"]]

    V_night_count = len(night_df)

    clusters = df_proc[cluster_col].unique().tolist()

    residence_scores: Dict[object, float] = {}
    work_scores: Dict[object, float] = {}
    notes = []

    # Helper to get unique valid day labels
    def _unique_valid_days(series: pd.Series) -> set:
        return set(d for d in series.unique() if d in VALID_DAYS)

    # Residence scores
    for c in clusters:
        vj_night = night_df[night_df[cluster_col] == c]
        days_vj_night = _unique_valid_days(vj_night[day_col]) if not vj_night.empty else set()
        consistency = len(days_vj_night) / 7.0
        frequency = (len(vj_night) / V_night_count) if V_night_count > 0 else 0.0
        residence_scores[c] = 0.5 * consistency + 0.5 * frequency

    if V_night_count == 0:
        notes.append("No night visits found; frequency term for residence = 0.")

    # Select predicted residence
    predicted_residence: Optional[object] = None
    if residence_scores:
        if V_night_count == 0 and all(score == 0 for score in residence_scores.values()):
            predicted_residence = None
            notes.append("All residence scores zero; predicted_residence set to None.")
        else:
            predicted_residence = max(residence_scores.items(), key=lambda x: x[1])[0]

    return {
        "residence_scores": {k: float(v) for k, v in residence_scores.items()},
        "work_scores": {k: float(v) for k, v in work_scores.items()},
        "predicted_residence": predicted_residence,
        "notes": "; ".join(notes) if notes else None,
    }

In [ ]:
from pathlib import Path
import sys, os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, parent_dir)

from clustering_polygons import cluster_polygons_boundary_threshold
park = "punggol"

geojson_dir = Path(f"../../gis_files/{park}/within_{park}_geojson")
geojson_files = [p.as_posix() for p in sorted(geojson_dir.glob("*.geojson"))]

In [ ]:
year = 2022

data_predicted = pd.DataFrame(columns=["id", "residence_x", "residence_y"])

for gjson in geojson_files:
    convex_hulls = gpd.read_file(gjson)
    did = gjson.split("/")[-1].split(".")[0].replace("clusters_convex_hulls_", "")

    # ensure datetimelike dtype
    convex_hulls['start_time'] = pd.to_datetime(convex_hulls['start_time'], errors='coerce')
    convex_hulls['end_time'] = pd.to_datetime(convex_hulls['end_time'], errors='coerce')

    convex_hulls['activity_start_date'] = convex_hulls['start_time'].dt.date
    convex_hulls['activity_end_date'] = convex_hulls['end_time'].dt.date

    # positive days crossed (0 if same day)
    convex_hulls['cross_day'] = (
        convex_hulls['end_time'].dt.normalize() - convex_hulls['start_time'].dt.normalize()
    ).dt.days

    convex_hulls['activity_date'] = [majority_day(s, e) for s, e in zip(convex_hulls['start_time'], convex_hulls['end_time'])]

    # Assuming df['activity_date'] is already created
    convex_hulls['activity_dayofweek'] = pd.to_datetime(convex_hulls['activity_date']).dt.day_name()

    # If you want for start/end too
    convex_hulls['start_dayofweek'] = convex_hulls['start_time'].dt.day_name()
    convex_hulls['end_dayofweek']   = convex_hulls['end_time'].dt.day_name()

    BINS = [
        ("Nighttime", time(0, 0),  time(7, 0)),
        ("Morning",   time(7, 0),  time(11, 0)),
        ("Afternoon", time(11, 0), time(16, 0)),
        ("Evening",   time(16, 0), time(22, 0)),
        ("Nighttime", time(22, 0), time(23, 59, 59, 999999)),
    ]

    def bin_for_instant(ts: pd.Timestamp) -> str:
        t = ts.time()
        for name, t0, t1 in BINS:
            if t0 <= t <= t1:
                return name
        return "Nighttime"  # fallback (should not hit)

    def majority_bin(start: pd.Timestamp, end: pd.Timestamp) -> str:
        # Tally durations per bin across all covered days
        tallies = {"Morning": pd.Timedelta(0),
                "Afternoon": pd.Timedelta(0),
                "Evening": pd.Timedelta(0),
                "Nighttime": pd.Timedelta(0)}

        cur_day = start.normalize()
        last_day = end.normalize()

        while cur_day <= last_day:
            day_start = cur_day
            day_end = cur_day + pd.Timedelta(days=1)

            # Overlap of activity with this day
            a = max(start, day_start)
            b = min(end,   day_end)
            if a < b:
                # Distribute into within-day bins
                for name, t0, t1 in BINS:
                    w0 = pd.Timestamp.combine(cur_day.date(), t0)
                    w1 = pd.Timestamp.combine(cur_day.date(), t1)
                    # Make the upper bound exclusive by a microsecond equivalent
                    w1 = w1 + pd.Timedelta(microseconds=1)
                    x0 = max(a, w0)
                    x1 = min(b, w1)
                    if x0 < x1:
                        tallies[name] += (x1 - x0)

            cur_day += pd.Timedelta(days=1)

        # Tie-break: favour the bin containing the start time
        start_bin = bin_for_instant(start)
        best = max(tallies.items(), key=lambda kv: (kv[1], kv[0] == start_bin))
        return best[0]

    convex_hulls['activity_tod'] = [majority_bin(s, e) for s, e in zip(convex_hulls['start_time'], convex_hulls['end_time'])]

    # (Optional) make it an ordered category
    convex_hulls['activity_tod'] = pd.Categorical(
        convex_hulls['activity_tod'],
        categories=["Morning", "Afternoon", "Evening", "Nighttime"],
        ordered=True
    )

    convex_hulls = cluster_polygons_boundary_threshold(convex_hulls, eps_m=50, include_touching=True, crs_m=3414)
    to_predict = convex_hulls[['geometry', 'activity_dayofweek', 'activity_tod', 'cluster_id']]

    result = predict_residence_and_workplace_from_tod(to_predict)
    predicted_residence = to_predict[to_predict['cluster_id'] == result['predicted_residence']]

    residence_x = predicted_residence.geometry.centroid.x.mean()
    residence_y = predicted_residence.geometry.centroid.y.mean()

    data_predicted.loc[len(data_predicted)] = {
        "id": did,
        "residence_x": residence_x,
        "residence_y": residence_y,
    }

In [ ]:
data_predicted.to_csv(f"predicted_locations_{park}_{year}.csv", index=False)
data_predicted_residence = data_predicted[['id', 'residence_x', 'residence_y']]

gpd_residence = gpd.GeoDataFrame(data_predicted_residence, geometry=gpd.points_from_xy(data_predicted_residence.residence_x, data_predicted_residence.residence_y))

# assign crs to 4326
gpd_residence.set_crs(epsg=4326, inplace=True)

# save to file
gpd_residence.to_file(f"predicted_residence_{park}_{year}.geojson", driver='GeoJSON')